# Stock Market Data Cleaning and Transformation Pipeline

## Project Overview:
This notebook implements a robust data pipeline for fetching, cleaning, transforming, and validating stock market data from Yahoo Finance. The primary goal is to prepare raw OHLCV (Open, High, Low, Close, Volume) data for further quantitative analysis, ensuring data quality, consistency, and an analysis-friendly structure.

### Key Features:
- Data extraction from Yahoo Finance using the `yfinance` library.
- Comprehensive data cleaning, including missing value imputation and duplicate handling.
- Transformation of multi-indexed, wide-format data into a vertical, long format for enhanced readability and analytical flexibility.
- Datatype optimization to reduce memory footprint and improve computational efficiency.
- Export of both raw and cleaned datasets for traceability and subsequent use.

### Project Details:
- **Owner:** Sushant S
- **Project:** Market Time Series Forecasting

## Table of Contents

1.  [Project Configuration](#project_configuration)
2.  [Install & Import Libraries](#install_import_libraries)
3.  [Fetch Data from Yahoo Finance](#fetch_data)
4.  [Save Raw Dataset](#save_raw_dataset)
5.  [Initial Dataset Inspection](#initial_inspection)
6.  [Restructure Dataset (Horizontal → Vertical)](#restructure_dataset)
7.  [Missing Value Analysis & Handling](#missing_value_handling)
8.  [Duplicate Analysis & Handling](#duplicate_handling)
9.  [Data Cleaning & Transformation](#data_cleaning_transformation)
10. [Datatype Optimization](#datatype_optimization)
11. [Final Validation](#final_validation)
12. [Export Cleaned Dataset](#export_cleaned_dataset)
13. [Business Summary / Conclusion](#business_summary)

<a id='install_import_libraries'></a>
## 1. Install & Import Libraries
This section ensures all necessary Python libraries are installed and imported for the project. It conditionally installs `yfinance` if it's not already present in the environment.

In [169]:
import os
import pandas as pd
import numpy as np
import datetime as dt

# Install yfinance if not already installed
try:
    import yfinance as yf
    print("yfinance is already installed.")
except ImportError:
    print("yfinance not found, installing...")
    !pip install yfinance
    import yfinance as yf
    print("yfinance installed successfully.")

print("All required libraries imported successfully.")

yfinance is already installed.
All required libraries imported successfully.


<a id='project_configuration'></a>
## 2. Project Configuration
Defines key parameters for data fetching and processing, ensuring easy modification and consistency across the notebook.

In [170]:
# List of stock tickers to fetch
STOCKS = ['AAPL', 'MSFT', 'SPY']

# Time range for data fetching (5 years ago from today)
END_DATE = dt.datetime.now()
START_DATE = END_DATE - dt.timedelta(days=5*365) # Approximately 5 years

# File paths for raw and cleaned data exports
RAW_DATA_PATH = 'raw_stock_data.csv'
CLEANED_DATA_PATH = 'cleaned_stock_data.csv'

print(f"Stocks to fetch: {STOCKS}")
print(f"Data start date: {START_DATE.strftime('%Y-%m-%d')}")
print(f"Data end date: {END_DATE.strftime('%Y-%m-%d')}")
print(f"Raw data will be saved to: {RAW_DATA_PATH}")
print(f"Cleaned data will be saved to: {CLEANED_DATA_PATH}")

Stocks to fetch: ['AAPL', 'MSFT', 'SPY']
Data start date: 2021-07-05
Data end date: 2026-07-04
Raw data will be saved to: raw_stock_data.csv
Cleaned data will be saved to: cleaned_stock_data.csv


<a id='fetch_data'></a>
## 3. Fetch Data from Yahoo Finance
This section fetches historical OHLCV (Open, High, Low, Close, Volume) data for the specified stocks and time range using the `yfinance` library. `auto_adjust=True` ensures adjusted prices are fetched and assigned directly to 'Open', 'High', 'Low', 'Close' columns.

In [171]:
try:
    # Fetch data, explicitly setting auto_adjust=False to get both 'Close' and 'Adj Close'
    raw_df = yf.download(STOCKS, start=START_DATE, end=END_DATE, auto_adjust=False)
    print(f"Successfully fetched data for {STOCKS} from {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}.")
except Exception as e:
    print(f"Error fetching data: {e}")
    raw_df = pd.DataFrame() # Create an empty DataFrame to avoid errors later

[*********************100%***********************]  3 of 3 completed

Successfully fetched data for ['AAPL', 'MSFT', 'SPY'] from 2021-07-05 to 2026-07-04.


<a id='save_raw_dataset'></a>
## 4. Save Raw Dataset
The fetched raw data is immediately saved to a CSV file for archiving and to ensure data traceability. This step preserves the original data before any cleaning or transformations are applied.

In [172]:
if not raw_df.empty:
    try:
        raw_df.to_csv(RAW_DATA_PATH)
        print(f"Raw data saved to {RAW_DATA_PATH}")
    except Exception as e:
        print(f"Error saving raw data: {e}")
else:
    print("Raw DataFrame is empty, skipping saving.")

Raw data saved to raw_stock_data.csv


<a id='initial_inspection'></a>
## 5. Initial Dataset Inspection
Perform initial checks on the raw dataset to understand its structure, identify potential issues, and verify data types, missing values, and descriptive statistics. A working copy of the DataFrame (`cleaned_df`) is created for subsequent processing.

### 5.1 Raw Data Head
Displays the first few rows of the raw dataset to get an initial glance at the data.

In [173]:
if not raw_df.empty:
    print("Raw Data Head:")
    display(raw_df.head())
else:
    print("Raw DataFrame is empty, skipping head display.")

Raw Data Head:


Price        Adj Close                               Close              \
Ticker            AAPL        MSFT         SPY        AAPL        MSFT   
Date                                                                     
2021-07-06  138.434280  266.469360  404.629639  142.020004  277.660004   
2021-07-07  140.919907  268.647888  406.059662  144.570007  279.929993   
2021-07-08  139.623459  266.239105  402.751099  143.240005  277.420013   
2021-07-09  141.446228  266.738098  407.050415  145.110001  277.940002   
2021-07-12  140.851639  266.143066  408.508362  144.500000  277.320007   

Price                         High                                 Low  \
Ticker             SPY        AAPL        MSFT         SPY        AAPL   
Date                                                                     
2021-07-06  432.929993  143.149994  279.369995  434.010010  140.070007   
2021-07-07  434.459991  144.889999  280.690002  434.760010  142.660004   
2021-07-08  430.920013  144.059998  278.730011  431.730011  140.669998   
2021-07-09  435.519989  145.649994  278.049988  435.839996  142.649994   
2021-07-12  437.079987  146.320007  279.769989  437.350006  144.000000   

Price                                     Open                          \
Ticker            MSFT         SPY        AAPL        MSFT         SPY   
Date                                                                     
2021-07-06  274.299988  430.010010  140.070007  278.029999  433.779999   
2021-07-07  277.149994  431.510010  143.539993  279.399994  433.660004   
2021-07-08  274.869995  427.519989  141.580002  276.899994  428.779999   
2021-07-09  275.320007  430.709991  142.750000  275.720001  432.529999   
2021-07-12  276.579987  434.970001  146.210007  279.160004  435.429993   

Price          Volume                      
Ticker           AAPL      MSFT       SPY  
Date                                       
2021-07-06  108181800  31565600  68710400  
2021-07-07  104911600  23260000  63549500  
2021-07-08  105575500  24618600  97595200  
2021-07-09   99890800  23916700  76238600  
2021-07-12   76299700  18931700  52889600

### 5.2 Raw Data Shape
Shows the number of rows and columns in the raw dataset.

In [174]:
if not raw_df.empty:
    print("Raw Data Shape:", raw_df.shape)
else:
    print("Raw DataFrame is empty, skipping shape display.")

Raw Data Shape: (1254, 18)


### 5.3 Raw Data Info
Provides a concise summary of the DataFrame, including column dtypes and non-null values. Essential for identifying initial data quality issues.

In [175]:
if not raw_df.empty:
    print("Raw Data Info:")
    raw_df.info()
else:
    print("Raw DataFrame is empty, skipping info display.")

Raw Data Info:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1254 entries, 2021-07-06 to 2026-07-02
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   (Adj Close, AAPL)  1254 non-null   float64
 1   (Adj Close, MSFT)  1254 non-null   float64
 2   (Adj Close, SPY)   1254 non-null   float64
 3   (Close, AAPL)      1254 non-null   float64
 4   (Close, MSFT)      1254 non-null   float64
 5   (Close, SPY)       1254 non-null   float64
 6   (High, AAPL)       1254 non-null   float64
 7   (High, MSFT)       1254 non-null   float64
 8   (High, SPY)        1254 non-null   float64
 9   (Low, AAPL)        1254 non-null   float64
 10  (Low, MSFT)        1254 non-null   float64
 11  (Low, SPY)         1254 non-null   float64
 12  (Open, AAPL)       1254 non-null   float64
 13  (Open, MSFT)       1254 non-null   float64
 14  (Open, SPY)        1254 non-null   float64
 15  (Volume, AAPL)     1254 non-null   int6

### 5.4 Raw Data Description
Generates descriptive statistics that summarize the central tendency, dispersion, and shape of the dataset's distribution.

In [176]:
if not raw_df.empty:
    print("Raw Data Description:")
    display(raw_df.describe())
else:
    print("Raw DataFrame is empty, skipping describe display.")

Raw Data Description:


Price     Adj Close                                  Close               \
Ticker         AAPL         MSFT          SPY         AAPL         MSFT   
count   1254.000000  1254.000000  1254.000000  1254.000000  1254.000000   
mean     193.983484   358.903651   500.031988   195.971962   365.405231   
std       44.911216    83.614717   112.059714    44.017036    81.678757   
min      122.933540   207.733994   339.378479   125.019997   214.250000   
25%      156.664875   283.036270   407.066757   159.837498   294.185013   
50%      182.720085   367.166107   459.891556   184.964996   371.949997   
75%      226.569931   418.292999   586.679611   228.027500   423.364998   
max      315.200012   538.658569   757.618225   315.200012   542.070007   

Price                       High                                    Low  \
Ticker          SPY         AAPL         MSFT          SPY         AAPL   
count   1254.000000  1254.000000  1254.000000  1254.000000  1254.000000   
mean     514.803429   197.933597   368.831922   517.551755   193.846539   
std      105.694776    44.389882    81.980070   105.762395    43.660515   
min      356.559998   127.769997   220.410004   359.820007   124.169998   
25%      429.547508   162.180000   295.482498   432.329994   157.802502   
50%      475.904999   186.655006   374.440002   477.044998   182.864998   
75%      596.975006   230.102501   426.842506   599.625015   225.799999   
max      759.570007   317.399994   555.450012   760.400024   309.649994   

Price                                    Open                            \
Ticker         MSFT          SPY         AAPL         MSFT          SPY   
count   1254.000000  1254.000000  1254.000000  1254.000000  1254.000000   
mean     361.734609   511.593229   195.776882   365.417647   514.690566   
std       81.387989   105.508111    44.011752    81.801343   105.708127   
min      213.429993   348.109985   126.010002   217.550003   349.209991   
25%      290.004997   426.019989   159.909996   292.952507   429.354988   
50%      367.400009   473.179993   184.899994   371.529999   475.350006   
75%      418.970009   594.177490   228.059998   423.827507   597.357498   
max      540.770020   756.750000   314.179993   555.229980   758.150024   

Price         Volume                              
Ticker          AAPL          MSFT           SPY  
count   1.254000e+03  1.254000e+03  1.254000e+03  
mean    6.530999e+07  2.657134e+07  7.571184e+07  
std     2.874114e+07  1.206951e+07  2.909854e+07  
min     1.791060e+07  5.855900e+06  2.604870e+07  
25%     4.578078e+07  1.906325e+07  5.663625e+07  
50%     5.728620e+07  2.364530e+07  7.120585e+07  
75%     7.733735e+07  3.073432e+07  8.941912e+07  
max     3.186799e+08  1.862016e+08  2.566114e+08

### 5.5 Missing Values in Raw Data (Counts)
Calculates the count of missing values for each column in the raw dataset. This helps prioritize cleaning efforts.

In [177]:
if not raw_df.empty:
    print("Missing Values (Counts) in Raw Data:")
    print(raw_df.isnull().sum())
else:
    print("Raw DataFrame is empty, skipping missing values count.")

Missing Values (Counts) in Raw Data:
Price      Ticker
Adj Close  AAPL      0
           MSFT      0
           SPY       0
Close      AAPL      0
           MSFT      0
           SPY       0
High       AAPL      0
           MSFT      0
           SPY       0
Low        AAPL      0
           MSFT      0
           SPY       0
Open       AAPL      0
           MSFT      0
           SPY       0
Volume     AAPL      0
           MSFT      0
           SPY       0
dtype: int64


### 5.6 Missing Values in Raw Data (Percentages)
Calculates the percentage of missing values for each column, offering a relative perspective on data completeness.

In [178]:
if not raw_df.empty:
    print("Missing Values (Percentages) in Raw Data:")
    print(raw_df.isnull().sum() / len(raw_df) * 100)
else:
    print("Raw DataFrame is empty, skipping missing values percentage.")

Missing Values (Percentages) in Raw Data:
Price      Ticker
Adj Close  AAPL      0.0
           MSFT      0.0
           SPY       0.0
Close      AAPL      0.0
           MSFT      0.0
           SPY       0.0
High       AAPL      0.0
           MSFT      0.0
           SPY       0.0
Low        AAPL      0.0
           MSFT      0.0
           SPY       0.0
Open       AAPL      0.0
           MSFT      0.0
           SPY       0.0
Volume     AAPL      0.0
           MSFT      0.0
           SPY       0.0
dtype: float64


### 5.7 Duplicated Index Check in Raw Data
Checks if there are any duplicate dates in the raw DataFrame's index. For time series, a unique index (date) is typically expected.

In [179]:
if not raw_df.empty:
    print("Duplicated Index (Dates) in Raw Data:")
    print(f"Number of duplicated dates: {raw_df.index.duplicated().sum()}")
else:
    print("Raw DataFrame is empty, skipping duplicate index check.")

Duplicated Index (Dates) in Raw Data:
Number of duplicated dates: 0


### 5.8 Create Working DataFrame Copy
Creates a copy of the raw DataFrame (`cleaned_df`) to perform cleaning and transformations, preserving the original raw data.

In [180]:
if not raw_df.empty:
    cleaned_df = raw_df.copy()
    print("Working DataFrame `cleaned_df` created from `raw_df`.")
else:
    cleaned_df = pd.DataFrame()
    print("Raw DataFrame is empty, `cleaned_df` initialized as empty.")

Working DataFrame `cleaned_df` created from `raw_df`.


<a id='restructure_dataset'></a>
## 6. Restructure Dataset (Horizontal → Vertical/Long Format)
This critical step transforms the wide-format, MultiIndex DataFrame (as returned by `yfinance`) into a normalized, vertical (long) format. This structure is more suitable for analysis, filtering, and visualization, adhering to best practices for time series data. Each row will represent a single stock's metric on a given date.

### 6.1 Flatten MultiIndex Columns
`yfinance` returns a MultiIndex DataFrame where columns are structured as `('Metric', 'Ticker')` (e.g., `('Close', 'AAPL')`). This step flattens these into single-level column names like `Metric_Ticker` (e.g., `Close_AAPL`).

In [181]:
if not cleaned_df.empty:
    new_columns = []
    # Check if columns are a MultiIndex
    if isinstance(cleaned_df.columns, pd.MultiIndex):
        if cleaned_df.columns.nlevels == 3:
            # Assumed structure: (level0, metric, ticker)
            for col_tuple in cleaned_df.columns.values:
                # Take the metric and ticker, replace spaces in metric with underscores
                metric = col_tuple[1].replace(' ', '_')
                ticker = col_tuple[2]
                new_columns.append(f"{metric}_{ticker}")
        elif cleaned_df.columns.nlevels == 2:
            # Assumed structure: (metric, ticker)
            for col_tuple in cleaned_df.columns.values:
                metric = col_tuple[0].replace(' ', '_')
                ticker = col_tuple[1]
                new_columns.append(f"{metric}_{ticker}")
        else:
            # Fallback for other MultiIndex levels - flatten all levels with underscore
            for col_tuple in cleaned_df.columns.values:
                new_columns.append('_'.join(map(str, col_tuple)).replace(' ', '_'))
    else:
        # Not a MultiIndex, just replace spaces in column names
        new_columns = [col.replace(' ', '_') for col in cleaned_df.columns]

    cleaned_df.columns = new_columns
    print("MultiIndex columns flattened. Sample columns:")
    print(cleaned_df.columns.tolist()[:10]) # Display first 10 flattened columns
else:
    print("DataFrame is empty, skipping column flattening.")

MultiIndex columns flattened. Sample columns:
['Adj_Close_AAPL', 'Adj_Close_MSFT', 'Adj_Close_SPY', 'Close_AAPL', 'Close_MSFT', 'Close_SPY', 'High_AAPL', 'High_MSFT', 'High_SPY', 'Low_AAPL']


### 6.2 Reset Index and Rename Date Column
The `Date` (index) is converted into a regular column, and ensures its name is consistent for further operations.

In [182]:
if not cleaned_df.empty:
    cleaned_df = cleaned_df.reset_index().rename(columns={'Date': 'Date'})
    print("Index reset, 'Date' column is now a regular column. Head of DataFrame:")
    display(cleaned_df.head())
else:
    print("DataFrame is empty, skipping index reset.")

Index reset, 'Date' column is now a regular column. Head of DataFrame:


,Date,Adj_Close_AAPL,Adj_Close_MSFT,Adj_Close_SPY,Close_AAPL,Close_MSFT,Close_SPY,High_AAPL,High_MSFT,High_SPY,Low_AAPL,Low_MSFT,Low_SPY,Open_AAPL,Open_MSFT,Open_SPY,Volume_AAPL,Volume_MSFT,Volume_SPY
0,2021-07-06,138.434280,266.469360,404.629639,142.020004,277.660004,432.929993,143.149994,279.369995,434.010010,140.070007,274.299988,430.010010,140.070007,278.029999,433.779999,108181800,31565600,68710400
1,2021-07-07,140.919907,268.647888,406.059662,144.570007,279.929993,434.459991,144.889999,280.690002,434.760010,142.660004,277.149994,431.510010,143.539993,279.399994,433.660004,104911600,23260000,63549500
2,2021-07-08,139.623459,266.239105,402.751099,143.240005,277.420013,430.920013,144.059998,278.730011,431.730011,140.669998,274.869995,427.519989,141.580002,276.899994,428.779999,105575500,24618600,97595200
3,2021-07-09,141.446228,266.738098,407.050415,145.110001,277.940002,435.519989,145.649994,278.049988,435.839996,142.649994,275.320007,430.709991,142.750000,275.720001,432.529999,99890800,23916700,76238600
4,2021-07-12,140.851639,266.143066,408.508362,144.500000,277.320007,437.079987,146.320007,279.769989,437.350006,144.000000,276.579987,434.970001,146.210007,279.160004,435.429993,76299700,18931700,52889600


### 6.3 Melt DataFrame to Long Format
This is the core transformation step. The wide-format DataFrame (e.g., `Open_AAPL`, `Open_MSFT`, `Open_SPY`) is `melted` into a long format where `Stock_Name` becomes a new column, and all corresponding metrics (`Open`, `High`, `Low`, `Close`, `Volume`) are aligned vertically per stock and date.

In [183]:
if not cleaned_df.empty:
    # Define the base columns (metrics) that exist in the flattened MultiIndex
    # Now includes both 'Close' (unadjusted) and 'Adj_Close' (adjusted), matching flattened names
    original_base_cols = ['Open', 'High', 'Low', 'Close', 'Adj_Close', 'Volume']

    melted_dfs = []
    for col_type in original_base_cols:
        cols_to_melt = [col for col in cleaned_df.columns if col.startswith(col_type + '_')]

        if cols_to_melt:
            temp_df = cleaned_df[['Date'] + cols_to_melt].copy()
            temp_melted = temp_df.melt(id_vars=['Date'], var_name='Stock_Metric', value_name=col_type)
            temp_melted['Stock_Name'] = temp_melted['Stock_Metric'].apply(lambda x: x.split('_')[-1])
            temp_melted = temp_melted.drop(columns=['Stock_Metric'])
            melted_dfs.append(temp_melted)

    if melted_dfs:
        final_cleaned_df = melted_dfs[0]
        for i in range(1, len(melted_dfs)):
            final_cleaned_df = pd.merge(final_cleaned_df, melted_dfs[i], on=['Date', 'Stock_Name'], how='outer')
    else:
        final_cleaned_df = pd.DataFrame()
        print("No data to melt after flattening columns.")

    cleaned_df = final_cleaned_df.copy()
    print("DataFrame melted to long format. Head of interim DataFrame:")
    display(cleaned_df.head())
else:
    print("DataFrame is empty, skipping melt operation.")

DataFrame melted to long format. Head of interim DataFrame:


,Date,Open,Stock_Name,High,Low,Close,Adj_Close,Volume
0,2021-07-06,140.070007,AAPL,143.149994,140.070007,142.020004,138.434280,108181800
1,2021-07-06,278.029999,MSFT,279.369995,274.299988,277.660004,266.469360,31565600
2,2021-07-06,433.779999,SPY,434.010010,430.010010,432.929993,404.629639,68710400
3,2021-07-07,143.539993,AAPL,144.889999,142.660004,144.570007,140.919907,104911600
4,2021-07-07,279.399994,MSFT,280.690002,277.149994,279.929993,268.647888,23260000


### 6.4 Rename 'Close' to 'Adj_Close'
Since `yfinance` with `auto_adjust=True` provides the adjusted closing prices under the 'Close' column, we explicitly rename it to 'Adj_Close' to reflect its nature and match the desired output schema.

In [184]:
if not cleaned_df.empty and 'Adj Close' in cleaned_df.columns:
    cleaned_df = cleaned_df.rename(columns={'Adj Close': 'Adj_Close'})
    print("'Adj Close' column renamed to 'Adj_Close'. Columns are now:")
    print(cleaned_df.columns.tolist())
elif not cleaned_df.empty and 'Adj_Close' not in cleaned_df.columns:
    print("Warning: 'Adj Close' column not found for renaming. Ensure `auto_adjust=False` was used during data fetch.")
else:
    print("DataFrame is empty or 'Adj Close' column already renamed/missing, skipping rename.")

DataFrame is empty or 'Adj Close' column already renamed/missing, skipping rename.


### 6.5 Sort Data for Consistency
The DataFrame is sorted by `Date` and `Stock_Name` to ensure a consistent and logical order for time series analysis.

In [185]:
if not cleaned_df.empty:
    cleaned_df = cleaned_df.sort_values(by=['Date', 'Stock_Name']).reset_index(drop=True)
    print("DataFrame sorted by Date and Stock_Name. Head of sorted DataFrame:")
    display(cleaned_df.head())
    print("Shape of cleaned DataFrame after restructuring:", cleaned_df.shape)
else:
    print("DataFrame is empty, skipping sort operation.")

DataFrame sorted by Date and Stock_Name. Head of sorted DataFrame:


,Date,Open,Stock_Name,High,Low,Close,Adj_Close,Volume
0,2021-07-06,140.070007,AAPL,143.149994,140.070007,142.020004,138.434280,108181800
1,2021-07-06,278.029999,MSFT,279.369995,274.299988,277.660004,266.469360,31565600
2,2021-07-06,433.779999,SPY,434.010010,430.010010,432.929993,404.629639,68710400
3,2021-07-07,143.539993,AAPL,144.889999,142.660004,144.570007,140.919907,104911600
4,2021-07-07,279.399994,MSFT,280.690002,277.149994,279.929993,268.647888,23260000


Shape of cleaned DataFrame after restructuring: (3762, 8)


<a id='missing_value_handling'></a>
## 7. Missing Value Analysis & Handling
This section systematically identifies and addresses missing values in the restructured dataset. For financial time series, a common strategy is forward-filling for price data (assuming the last known price persists) and filling volume with zeros or a more sophisticated imputation.

### 7.1 Missing Values Before Handling (Counts)
Shows the count of `NaN` values per column prior to any imputation.

In [186]:
if not cleaned_df.empty:
    print("Missing values before handling (Counts):\n", cleaned_df.isnull().sum())
else:
    print("DataFrame is empty, skipping missing value count.")

Missing values before handling (Counts):
 Date          0
Open          0
Stock_Name    0
High          0
Low           0
Close         0
Adj_Close     0
Volume        0
dtype: int64


### 7.2 Missing Values Before Handling (Percentages)
Provides the percentage of `NaN` values per column before handling.

In [187]:
if not cleaned_df.empty:
    print("Missing values before handling (Percentages):\n", cleaned_df.isnull().sum() / len(cleaned_df) * 100)
else:
    print("DataFrame is empty, skipping missing value percentage.")

Missing values before handling (Percentages):
 Date          0.0
Open          0.0
Stock_Name    0.0
High          0.0
Low           0.0
Close         0.0
Adj_Close     0.0
Volume        0.0
dtype: float64


### 7.3 Handle Missing Values: Forward-Fill Prices & Zero-Fill Volume
- Price-related columns (`Open`, `High`, `Low`, `Adj_Close`) are forward-filled (`ffill`) grouped by `Stock_Name`. This assumes that if a price is missing for a day, the previous day's price holds.
- The `Volume` column is filled with `0` for missing values, as a missing volume might indicate no trading activity for that stock on that day.

In [188]:
if not cleaned_df.empty:
    numeric_cols = cleaned_df.select_dtypes(include=np.number).columns.tolist()
    price_cols = [col for col in numeric_cols if col not in ['Volume']]
    volume_col = 'Volume'

    if price_cols:
        print(f"Forward filling missing values for price-related columns: {price_cols}")
        cleaned_df[price_cols] = cleaned_df.groupby('Stock_Name')[price_cols].ffill()

    if volume_col in cleaned_df.columns:
        print(f"Filling missing values for {volume_col} with 0.")
        cleaned_df[volume_col] = cleaned_df.groupby('Stock_Name')[volume_col].fillna(0)

    print("Missing values handling applied for prices and volume.")
else:
    print("DataFrame is empty, skipping missing value handling.")

Forward filling missing values for price-related columns: ['Open', 'High', 'Low', 'Close', 'Adj_Close']
Filling missing values for Volume with 0.
Missing values handling applied for prices and volume.


/tmp/ipykernel_2580/694252895.py:12: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  cleaned_df[volume_col] = cleaned_df.groupby('Stock_Name')[volume_col].fillna(0)


### 7.4 Drop Initial Missing Rows (Validation after Handling)
After forward-filling, some leading `NaN` values might remain if a stock's data starts with missing entries. This step drops such rows, ensuring no `NaN`s are left in the critical price columns, and then validates that all missing values have been addressed.

In [189]:
if not cleaned_df.empty:
    # Re-evaluate price_cols after initial fillna to ensure only relevant columns are checked
    numeric_cols = cleaned_df.select_dtypes(include=np.number).columns.tolist()
    price_cols = [col for col in numeric_cols if col not in ['Volume']]

    initial_nans_check = cleaned_df.groupby('Stock_Name')[price_cols].apply(lambda x: x.iloc[0].isnull().any())
    if initial_nans_check.any():
        print("Dropping initial rows with NaNs in price data for specific stocks...")
        cleaned_df = cleaned_df.dropna(subset=price_cols, how='all').reset_index(drop=True)

    print("Missing values after handling and dropping initial NaNs (Counts):\n", cleaned_df.isnull().sum())

    # Validation step
    if cleaned_df.isnull().sum().sum() == 0:
        print("Validation: All missing values handled successfully.")
    else:
        print("Validation Warning: Some missing values still exist. Re-evaluate strategy.")
else:
    print("DataFrame is empty, skipping initial missing rows drop and validation.")

Missing values after handling and dropping initial NaNs (Counts):
 Date          0
Open          0
Stock_Name    0
High          0
Low           0
Close         0
Adj_Close     0
Volume        0
dtype: int64
Validation: All missing values handled successfully.


<a id='duplicate_handling'></a>
## 8. Duplicate Analysis & Handling
This section checks for and handles duplicate rows. In a time series dataset structured in long format, a duplicate is typically defined by a unique combination of `Date` and `Stock_Name`. We aim to ensure that each stock has only one entry per day.

### 8.1 Check for Duplicates Before Handling
Identifies and counts any rows that are exact duplicates based on the `Date` and `Stock_Name` columns.

In [190]:
if not cleaned_df.empty:
    initial_duplicates = cleaned_df.duplicated(subset=['Date', 'Stock_Name']).sum()
    print(f"Number of duplicated (Date, Stock_Name) pairs before handling: {initial_duplicates}")
else:
    print("DataFrame is empty, skipping duplicate check.")

Number of duplicated (Date, Stock_Name) pairs before handling: 0


### 8.2 Remove Duplicates
If duplicates are found, this step removes them, keeping only the first occurrence based on `Date` and `Stock_Name`.

In [191]:
if not cleaned_df.empty and initial_duplicates > 0:
    print("Dropping duplicated rows based on 'Date' and 'Stock_Name'...")
    cleaned_df = cleaned_df.drop_duplicates(subset=['Date', 'Stock_Name'], keep='first').reset_index(drop=True)
    print("Duplicates dropped. Head of DataFrame after removal:")
    display(cleaned_df.head())
elif not cleaned_df.empty:
    print("No duplicates found, skipping removal.")
else:
    print("DataFrame is empty, skipping duplicate removal.")

No duplicates found, skipping removal.


### 8.3 Validate Duplicates After Handling
Re-checks for duplicates to confirm that the removal process was successful.

In [192]:
if not cleaned_df.empty:
    final_duplicates = cleaned_df.duplicated(subset=['Date', 'Stock_Name']).sum()
    print(f"Number of duplicated (Date, Stock_Name) pairs after handling: {final_duplicates}")

    if final_duplicates == 0:
        print("Validation: No duplicates found after handling.")
    else:
        print("Validation Warning: Duplicates still exist. Re-evaluate strategy.")
else:
    print("DataFrame is empty, skipping duplicate validation.")

Number of duplicated (Date, Stock_Name) pairs after handling: 0
Validation: No duplicates found after handling.


<a id='data_cleaning_transformation'></a>
## 9. Data Cleaning & Transformation: Datetime Conversion
This section specifically focuses on ensuring the `Date` column is of the correct datetime data type, which is crucial for time series analysis and operations.

### 9.1 Data Types Before Datetime Conversion
Displays the current data types of all columns before converting the `Date` column.

In [193]:
if not cleaned_df.empty:
    print("Data types before datetime conversion:\n", cleaned_df.dtypes)
else:
    print("DataFrame is empty, skipping dtypes display.")

Data types before datetime conversion:
 Date          datetime64[ns]
Open                 float64
Stock_Name            object
High                 float64
Low                  float64
Close                float64
Adj_Close            float64
Volume                 int64
dtype: object


### 9.2 Convert 'Date' Column to Datetime
Converts the `Date` column to `datetime64` format if it isn't already. This is essential for time-based indexing and operations.

In [194]:
if not cleaned_df.empty:
    if not pd.api.types.is_datetime64_any_dtype(cleaned_df['Date']):
        print("Converting 'Date' column to datetime format...")
        cleaned_df['Date'] = pd.to_datetime(cleaned_df['Date'])
        print("Date column converted. Sample dates:\n", cleaned_df['Date'].head().dt.strftime('%Y-%m-%d'))
    else:
        print("'Date' column is already in datetime format.")
else:
    print("DataFrame is empty, skipping date conversion.")

'Date' column is already in datetime format.


### 9.3 Data Types After Datetime Conversion (Validation)
Verifies that the `Date` column has been successfully converted to the correct data type.

In [195]:
if not cleaned_df.empty:
    print("Data types after datetime conversion:\n", cleaned_df.dtypes)
    if pd.api.types.is_datetime64_any_dtype(cleaned_df['Date']):
        print("Validation: 'Date' column successfully transformed to datetime type.")
    else:
        print("Validation Warning: 'Date' column type is incorrect.")
else:
    print("DataFrame is empty, skipping datetime conversion validation.")

Data types after datetime conversion:
 Date          datetime64[ns]
Open                 float64
Stock_Name            object
High                 float64
Low                  float64
Close                float64
Adj_Close            float64
Volume                 int64
dtype: object
Validation: 'Date' column successfully transformed to datetime type.


<a id='datatype_optimization'></a>
## 10. Datatype Optimization
This section optimizes numerical and categorical column data types to reduce memory usage and improve computational efficiency. For example, `float64` columns are downcasted to `float32` and `int64` to `int32` where safe, and `object` columns with limited unique values are converted to `category` type.

### 10.1 Memory Usage Before Optimization
Calculates and displays the memory footprint of the DataFrame before applying any datatype optimizations.

In [196]:
if not cleaned_df.empty:
    def get_memory_usage(df):
        return df.memory_usage(deep=True).sum() / (1024**2)

    initial_memory_mb = get_memory_usage(cleaned_df)
    print(f"Initial memory usage: {initial_memory_mb:.2f} MB")
else:
    print("DataFrame is empty, skipping memory usage calculation.")

Initial memory usage: 0.39 MB


### 10.2 Apply Datatype Conversions
Iterates through columns, converting float columns to `float32`, integer columns to `int32` (if safely possible without loss of information), and object columns with a low cardinality to `category` type.

In [197]:
if not cleaned_df.empty:
    print("Optimizing numerical and categorical dtypes...")
    for col in cleaned_df.columns:
        if cleaned_df[col].dtype == 'float64':
            min_val = cleaned_df[col].min()
            max_val = cleaned_df[col].max()
            if np.finfo(np.float32).min <= min_val and max_val <= np.finfo(np.float32).max:
                cleaned_df[col] = cleaned_df[col].astype(np.float32)
            else:
                print(f"Column '{col}' cannot be safely converted to float32 due to range limits.")
        elif cleaned_df[col].dtype == 'int64':
            min_val = cleaned_df[col].min()
            max_val = cleaned_df[col].max()
            if np.iinfo(np.int32).min <= min_val and max_val <= np.iinfo(np.int32).max:
                cleaned_df[col] = cleaned_df[col].astype(np.int32)
            else:
                print(f"Column '{col}' cannot be safely converted to int32 due to range limits.")
        elif cleaned_df[col].dtype == 'object' and col != 'Date': # Date is handled as datetime
            if cleaned_df[col].nunique() / len(cleaned_df[col]) < 0.5: # Heuristic for categorical
                cleaned_df[col] = cleaned_df[col].astype('category')
    print("Datatype optimization applied.")
else:
    print("DataFrame is empty, skipping datatype conversion.")

Optimizing numerical and categorical dtypes...
Datatype optimization applied.


### 10.3 Memory Usage After Optimization
Displays the memory footprint of the DataFrame after applying datatype optimizations.

In [198]:
if not cleaned_df.empty:
    optimized_memory_mb = get_memory_usage(cleaned_df)
    print(f"Optimized memory usage: {optimized_memory_mb:.2f} MB")
else:
    print("DataFrame is empty, skipping optimized memory usage calculation.")

Optimized memory usage: 0.12 MB


### 10.4 Memory Improvement Calculation (Validation)
Calculates the reduction in memory usage and its percentage improvement, along with a validation check.

In [199]:
if not cleaned_df.empty:
    if 'initial_memory_mb' in locals():
        memory_reduced = initial_memory_mb - optimized_memory_mb
        percentage_reduction = (memory_reduced / initial_memory_mb) * 100
        print(f"Memory reduced by: {memory_reduced:.2f} MB ({percentage_reduction:.2f}%)")
        if optimized_memory_mb < initial_memory_mb:
            print("Validation: Datatypes optimized and memory footprint reduced.")
        else:
            print("Validation Warning: Datatype optimization did not reduce memory or was not applied.")
    else:
        print("Initial memory usage not recorded, skipping memory improvement calculation.")
else:
    print("DataFrame is empty, skipping memory improvement calculation.")

Memory reduced by: 0.27 MB (69.54%)
Validation: Datatypes optimized and memory footprint reduced.


### 10.5 Data Types After Optimization (Validation)
Displays the final data types of all columns to confirm the optimizations were applied as expected.

In [200]:
if not cleaned_df.empty:
    print("Data types after optimization:\n", cleaned_df.dtypes)
else:
    print("DataFrame is empty, skipping final dtypes display.")

Data types after optimization:
 Date          datetime64[ns]
Open                 float32
Stock_Name          category
High                 float32
Low                  float32
Close                float32
Adj_Close            float32
Volume                 int32
dtype: object


<a id='final_validation'></a>
## 11. Final Validation
A final comprehensive check of the cleaned dataset to ensure all cleaning and transformation steps have been successfully applied and the data is fully ready for analysis. This section reiterates key data quality checks.

### 11.1 Final Cleaned Data Head
Displays the first few rows of the fully cleaned and transformed DataFrame.

In [201]:
if not cleaned_df.empty:
    print("Final Cleaned Data Head:")
    display(cleaned_df.head())
else:
    print("DataFrame is empty, skipping final head display.")

Final Cleaned Data Head:


,Date,Open,Stock_Name,High,Low,Close,Adj_Close,Volume
0,2021-07-06,140.070007,AAPL,143.149994,140.070007,142.020004,138.434280,108181800
1,2021-07-06,278.029999,MSFT,279.369995,274.299988,277.660004,266.469360,31565600
2,2021-07-06,433.779999,SPY,434.010010,430.010010,432.929993,404.629639,68710400
3,2021-07-07,143.539993,AAPL,144.889999,142.660004,144.570007,140.919907,104911600
4,2021-07-07,279.399994,MSFT,280.690002,277.149994,279.929993,268.647888,23260000


### 11.2 Final Cleaned Data Shape
Confirms the final dimensions of the cleaned dataset.

In [202]:
if not cleaned_df.empty:
    print("Final Cleaned Data Shape:", cleaned_df.shape)
else:
    print("DataFrame is empty, skipping final shape display.")

Final Cleaned Data Shape: (3762, 8)


### 11.3 Final Cleaned Data Info
Provides a final summary of the DataFrame's structure, including data types and non-null counts after all cleaning.

In [203]:
if not cleaned_df.empty:
    print("Final Cleaned Data Info:")
    cleaned_df.info()
else:
    print("DataFrame is empty, skipping final info display.")

Final Cleaned Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3762 entries, 0 to 3761
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        3762 non-null   datetime64[ns]
 1   Open        3762 non-null   float32       
 2   Stock_Name  3762 non-null   category      
 3   High        3762 non-null   float32       
 4   Low         3762 non-null   float32       
 5   Close       3762 non-null   float32       
 6   Adj_Close   3762 non-null   float32       
 7   Volume      3762 non-null   int32         
dtypes: category(1), datetime64[ns](1), float32(5), int32(1)
memory usage: 121.5 KB


### 11.4 Final Cleaned Data Description
Generates descriptive statistics for the numerical columns of the cleaned DataFrame.

In [204]:
if not cleaned_df.empty:
    print("Final Cleaned Data Description:")
    display(cleaned_df.describe())
else:
    print("DataFrame is empty, skipping final describe display.")

Final Cleaned Data Description:


,Date,Open,High,Low,Close,Adj_Close,Volume
count,3762,3762.000000,3762.000000,3762.000000,3762.000000,3762.000000,3.762000e+03
mean,2023-12-31 22:02:52.248803584,358.628387,361.439056,355.724823,358.726868,350.973022,5.586439e+07
min,2021-07-06 00:00:00,126.010002,127.769997,124.169998,125.019997,122.933540,5.855900e+06
25%,2022-09-30 00:00:00,227.347496,229.582497,225.379997,227.527504,225.401707,2.998885e+07
50%,2023-12-31 00:00:00,368.970001,373.039993,364.694992,368.684998,356.570236,5.023690e+07
75%,2025-04-02 00:00:00,450.730011,452.977509,448.727509,450.892502,434.204849,7.406888e+07
max,2026-07-02 00:00:00,758.150024,760.400024,756.750000,759.570007,757.618225,3.186799e+08
std,NaN,153.544571,153.882767,152.971649,153.489319,151.100510,3.245029e+07


### 11.5 Missing Values in Final Cleaned Data (Validation)
Ensures that no missing values remain in the dataset after all imputation and dropping steps.

In [205]:
if not cleaned_df.empty:
    print("Missing Values in Final Cleaned Data (Counts):\n", cleaned_df.isnull().sum())
    if cleaned_df.isnull().sum().sum() == 0:
        print("Status: No missing values detected.")
    else:
        print("Status: Missing values still present! Re-evaluate cleaning strategy.")
else:
    print("DataFrame is empty, skipping final missing values check.")

Missing Values in Final Cleaned Data (Counts):
 Date          0
Open          0
Stock_Name    0
High          0
Low           0
Close         0
Adj_Close     0
Volume        0
dtype: int64
Status: No missing values detected.


### 11.6 Duplicated Rows in Final Cleaned Data (Validation)
Confirms that no duplicate rows exist in the final dataset based on the `Date` and `Stock_Name` combination.

In [206]:
if not cleaned_df.empty:
    final_duplicates_count = cleaned_df.duplicated(subset=['Date', 'Stock_Name']).sum()
    print(f"Number of duplicated (Date, Stock_Name) pairs in Final Cleaned Data: {final_duplicates_count}")
    if final_duplicates_count == 0:
        print("Status: No duplicate (Date, Stock_Name) pairs detected.")
    else:
        print("Status: Duplicates still present! Re-evaluate duplicate handling strategy.")
else:
    print("DataFrame is empty, skipping final duplicate check.")

Number of duplicated (Date, Stock_Name) pairs in Final Cleaned Data: 0
Status: No duplicate (Date, Stock_Name) pairs detected.


### 11.7 Data Structure Check (Validation)
Validates that the final DataFrame has the exact required columns and structure (`Date | Stock_Name | Open | High | Low | Adj_Close | Volume`).

In [207]:
if not cleaned_df.empty:
    expected_cols = ['Date', 'Stock_Name', 'Open', 'High', 'Low', 'Close', 'Adj_Close', 'Volume']
    if set(cleaned_df.columns) == set(expected_cols) and len(cleaned_df.columns) == len(expected_cols):
        print("Status: Data structure matches the required long format.")
    else:
        print("Status: Data structure does NOT match the required long format. Check column names and count.")
        print("Expected columns:", sorted(expected_cols))
        print("Actual columns:", sorted(cleaned_df.columns.tolist()))
    print("Final validation completed.")
else:
    print("DataFrame is empty, skipping final data structure validation.")

Status: Data structure matches the required long format.
Final validation completed.


<a id='export_cleaned_dataset'></a>
## 12. Export Cleaned Dataset
The fully cleaned and transformed dataset is exported to a new CSV file (`cleaned_stock_data.csv`). This file is now ready for advanced analysis, visualization, or machine learning model training, ensuring data consistency and quality.

In [208]:
if not cleaned_df.empty:
    try:
        cleaned_df.to_csv(CLEANED_DATA_PATH, index=False)
        print(f"Cleaned data successfully exported to {CLEANED_DATA_PATH}")
    except Exception as e:
        print(f"Error exporting cleaned data: {e}")
else:
    print("Cleaned DataFrame is empty, skipping export.")

Cleaned data successfully exported to cleaned_stock_data.csv


## Download Exported Data

Use the following code to download the `raw_stock_data.csv` and `cleaned_stock_data.csv` files to your local machine.

In [209]:
from google.colab import files

# Download raw data
try:
    files.download(RAW_DATA_PATH)
    print(f"Downloading {RAW_DATA_PATH}...")
except Exception as e:
    print(f"Error downloading {RAW_DATA_PATH}: {e}")

# Download cleaned data
try:
    files.download(CLEANED_DATA_PATH)
    print(f"Downloading {CLEANED_DATA_PATH}...")
except Exception as e:
    print(f"Error downloading {CLEANED_DATA_PATH}: {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<a id='business_summary'></a>
## 13. Business Summary / Conclusion
This notebook successfully implemented an end-to-end data pipeline for stock market data, from raw extraction to a validated, analysis-ready format. The transformation to a vertical (long) dataset with optimized datatypes provides a robust foundation for subsequent quantitative analysis and predictive modeling initiatives, ensuring high data quality and efficient processing. This modular approach ensures auditability and maintainability for enterprise-grade financial data workflows.